# Gold — dashboard Superadmin
Execute depois de Bronze e Silver no mesmo catálogo/schema, com compute Python/PySpark. Fontes: silver_site_chamados, silver_site_usuarios, silver_site_controle_rpa e silver_site_qualidade.
A Gold mantém snapshots atuais em Delta. Cada execução substitui os dados anteriores, usando um mesmo atualizado_em e uma mesma versao_publicacao para todas as saídas. Não publica se houver registros Silver inválidos.


In [ ]:
from datetime import datetime, timezone
from uuid import uuid4
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")
dbutils.widgets.text("nome_carga_rpa", "carga_delta_legado")
NOME_CARGA_RPA = dbutils.widgets.get("nome_carga_rpa").strip()
if not NOME_CARGA_RPA:
    raise ValueError("Informe o nome real da carga RPA no widget nome_carga_rpa.")

ATUALIZADO_EM = datetime.now(timezone.utc)
VERSAO = str(uuid4())

qualidade = spark.table("silver_site_qualidade")
entidades = {"chamados", "usuarios", "controle_rpa"}
resumo = qualidade.collect()
if len(resumo) != 3 or {r["entidade"] for r in resumo} != entidades:
    raise RuntimeError("Resumo Silver incompleto ou duplicado. Execute todo o Silver.")
if any(r["invalidos"] is None or r["invalidos"] != 0 for r in resumo):
    raise RuntimeError("Há registros inválidos na Silver. Corrija-os antes da Gold.")

chamados = spark.table("silver_site_chamados")
usuarios = spark.table("silver_site_usuarios")
rpa = spark.table("silver_site_controle_rpa")

for entidade, df, chave in [
    ("chamados", chamados, "id"), ("usuarios", usuarios, "id"),
    ("controle_rpa", rpa, "nome_carga")
]:
    total = df.count()
    esperado = next(r["validos"] for r in resumo if r["entidade"] == entidade)
    if total != esperado or df.filter(F.col(chave).isNull()).limit(1).count():
        raise RuntimeError(f"Contagem ou chave inválida na Silver: {entidade}.")
    if df.groupBy(chave).count().filter("count > 1").limit(1).count():
        raise RuntimeError(f"Chave duplicada na Silver: {entidade}.")
    if spark.table(f"silver_site_{entidade}_invalidos").limit(1).count():
        raise RuntimeError(f"Quarentena não vazia: {entidade}.")


## Base detalhada
gold_site_chamados: uma linha por chamado. Preserva as duas datas e inclui nomes para exibição e perfil do solicitante para filtro, evitando joins na ferramenta de BI. Responsável ausente continua nulo; o rótulo SEM RESPONSAVEL é apenas de apresentação.


In [ ]:
solicitantes = usuarios.select(
    F.col("id").alias("solicitante_id"),
    F.col("nivel_acesso").alias("nivel_acesso_solicitante")
)
responsaveis = usuarios.select(
    F.col("id").alias("responsavel_id"), F.col("nome").alias("responsavel_nome")
)
detalhe = (chamados
    .join(solicitantes, chamados.cod_usuario_solicitante == solicitantes.solicitante_id, "left")
    .join(responsaveis, chamados.cod_usuario_responsavel == responsaveis.responsavel_id, "left")
    .select(
        F.col("id").alias("chamado_id"),
        "cod_usuario_solicitante", "nivel_acesso_solicitante",
        "cod_usuario_responsavel", "responsavel_nome",
        F.coalesce(F.col("responsavel_nome"), F.lit("SEM RESPONSAVEL")).alias("responsavel_rotulo"),
        "categoria", "status", "data_hora_abertura", "data_hora_encerramento"
    )
    .withColumn("data_abertura", F.to_date("data_hora_abertura"))
    .withColumn("data_encerramento", F.to_date("data_hora_encerramento"))
    .withColumn("quantidade_chamados", F.lit(1).cast("long"))
    .withColumn("quantidade_abertos", F.col("status").isin("REGISTRADO", "EM_ATENDIMENTO").cast("long"))
    .withColumn("quantidade_finalizados", (F.col("status") == "ATENDIDO").cast("long"))
    .withColumn("tempo_resolucao_horas", F.when(
        (F.col("status") == "ATENDIDO") &
        F.col("data_hora_encerramento").isNotNull() &
        (F.col("data_hora_encerramento") >= F.col("data_hora_abertura")),
        (F.col("data_hora_encerramento").cast("double") -
         F.col("data_hora_abertura").cast("double")) / F.lit(3600.0)
    ))
)
if detalhe.count() != chamados.count():
    raise RuntimeError("Os joins alteraram o grão de um registro por chamado.")
if detalhe.filter(
    F.col("nivel_acesso_solicitante").isNull() |
    ((F.col("status") == "ATENDIDO") & F.col("tempo_resolucao_horas").isNull()) |
    (F.col("tempo_resolucao_horas") < 0)
).limit(1).count():
    raise RuntimeError("Relacionamento ou duração inválida na base de chamados.")

def metricas_chamados(df, grupos=None):
    expressoes = [
        F.coalesce(F.sum("quantidade_chamados"), F.lit(0)).alias("total_chamados"),
        F.coalesce(F.sum("quantidade_abertos"), F.lit(0)).alias("chamados_abertos"),
        F.coalesce(F.sum("quantidade_finalizados"), F.lit(0)).alias("chamados_finalizados"),
        F.coalesce(F.sum("tempo_resolucao_horas"), F.lit(0.0)).alias("soma_tempo_resolucao_horas"),
        F.count("tempo_resolucao_horas").alias("quantidade_resolucoes"),
        F.avg("tempo_resolucao_horas").alias("tempo_medio_resolucao_horas")
    ]
    return df.groupBy(*grupos).agg(*expressoes) if grupos else df.agg(*expressoes)

# Grão reduzido, mas com todas as dimensões necessárias aos filtros combinados.
por_abertura = metricas_chamados(detalhe, [
    "data_abertura", "status", "categoria", "cod_usuario_responsavel",
    "responsavel_nome", "responsavel_rotulo", "nivel_acesso_solicitante"
])
kpis = metricas_chamados(detalhe)


## Evolução por período, usuários e RPA
Abertura e finalização usam datas de eventos diferentes. A série abaixo não representa o estoque histórico de chamados em aberto: esse histórico não existe nas fontes atuais.


In [ ]:
dimensoes = ["status", "categoria", "cod_usuario_responsavel",
             "responsavel_rotulo", "nivel_acesso_solicitante"]
aberturas = detalhe.select(
    F.col("data_abertura").alias("data_evento"), *dimensoes,
    F.lit(1).cast("long").alias("abertos_no_dia"),
    F.lit(0).cast("long").alias("finalizados_no_dia")
)
finalizacoes = detalhe.filter(F.col("status") == "ATENDIDO").select(
    F.col("data_encerramento").alias("data_evento"), *dimensoes,
    F.lit(0).cast("long").alias("abertos_no_dia"),
    F.lit(1).cast("long").alias("finalizados_no_dia")
)
evolucao = (aberturas.unionByName(finalizacoes)
    .groupBy("data_evento", *dimensoes)
    .agg(F.sum("abertos_no_dia").alias("abertos_no_dia"),
         F.sum("finalizados_no_dia").alias("finalizados_no_dia"))
)

# Inclui perfis sem usuários, com contagem zero.
perfis = spark.createDataFrame(
    [(p,) for p in ["OPERADOR", "GESTOR", "ADMIN", "SUPER_ADMIN"]], "nivel_acesso string")
usuarios_perfil = (perfis.join(
    usuarios.groupBy("nivel_acesso").agg(F.count("*").alias("total_usuarios")),
    "nivel_acesso", "left").fillna({"total_usuarios": 0})
)
usuarios_kpis = usuarios.agg(F.count("*").alias("total_usuarios"))

# Um registro para a carga escolhida, mesmo se nunca tiver executado.
carga = spark.createDataFrame([(NOME_CARGA_RPA,)], "nome_carga string")
ultima_carga = (carga.join(rpa.select(
    "nome_carga", "ultima_execucao_sucesso"), "nome_carga", "left"))

saidas = {
    "gold_site_chamados": detalhe,
    "gold_site_chamados_por_abertura": por_abertura,
    "gold_site_chamados_evolucao": evolucao,
    "gold_site_chamados_kpis": kpis,
    "gold_site_usuarios_por_nivel": usuarios_perfil,
    "gold_site_usuarios_kpis": usuarios_kpis,
    "gold_site_ultima_carga_rpa": ultima_carga,
}


In [ ]:
# Reconciliação antes de qualquer gravação Gold.
medidas = kpis.first()
if medidas["total_chamados"] != chamados.count():
    raise RuntimeError("Total Gold difere da Silver.")
if medidas["chamados_abertos"] + medidas["chamados_finalizados"] != medidas["total_chamados"]:
    raise RuntimeError("Status não classificado como aberto ou finalizado.")
totais_eventos = evolucao.agg(
    F.coalesce(F.sum("abertos_no_dia"), F.lit(0)).alias("aberturas"),
    F.coalesce(F.sum("finalizados_no_dia"), F.lit(0)).alias("finalizacoes")
).first()
if (totais_eventos["aberturas"] != medidas["total_chamados"] or
    totais_eventos["finalizacoes"] != medidas["chamados_finalizados"]):
    raise RuntimeError("Série de eventos não reconcilia com os KPIs.")
if usuarios_perfil.agg(F.sum("total_usuarios")).first()[0] != usuarios.count():
    raise RuntimeError("Perfis não reconciliam com o total de usuários.")

for nome, df in saidas.items():
    (df.withColumn("atualizado_em", F.lit(ATUALIZADO_EM))
       .withColumn("versao_publicacao", F.lit(VERSAO))
       .write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true").saveAsTable(nome))
    print(f"Gold salva: {nome}")

display(spark.table("gold_site_chamados_kpis"))
display(spark.table("gold_site_usuarios_kpis"))
display(spark.table("gold_site_ultima_carga_rpa"))


## Contrato das métricas e uso nos gráficos
| KPI / gráfico | Fonte e cálculo |
| --- | --- |
| Total de chamados filtrado por data | Somar total_chamados em gold_site_chamados_por_abertura, filtrando data_abertura |
| Cards de abertos / finalizados | Somar chamados_abertos / chamados_finalizados na mesma base filtrada; REGISTRADO e EM_ATENDIMENTO são abertos; ATENDIDO é finalizado |
| Barras por status, categoria ou responsável | Agrupar gold_site_chamados_por_abertura pela dimensão e somar total_chamados. Atendimentos por status refere-se ao suporte |
| Abertos e finalizados por período | Somar abertos_no_dia e finalizados_no_dia em gold_site_chamados_evolucao, filtrando data_evento. Agregar dia, semana ou mês no consumidor |
| Total de usuários / barras por perfil | gold_site_usuarios_kpis / gold_site_usuarios_por_nivel; snapshots atuais, sem filtro temporal de cadastro |
| Tempo médio de resolução filtrado | SUM(soma_tempo_resolucao_horas) / SUM(quantidade_resolucoes), retornando null se o denominador for zero |
| Histograma e boxplot de resolução | tempo_resolucao_horas dos registros ATENDIDO em gold_site_chamados; não usar médias agregadas para distribuições |
| Última carga RPA | ultima_execucao_sucesso de gold_site_ultima_carga_rpa para nome_carga_rpa; null significa ausência de sucesso registrado para essa carga |
| Relatórios detalhados | gold_site_chamados, com filtros e ordenação no consumidor |
Sem filtros, gold_site_chamados_kpis entrega uma linha com os cards gerais. Não usar esse snapshot de cards para responder a filtros de período, categoria ou responsável. Não calcular média das médias: usar sempre soma das durações dividida pela quantidade de resoluções.
O período de abertura seleciona uma coorte de chamados; o período de encerramento seleciona finalizações. Para um card de resolução por período de encerramento, consultar a base detalhada e filtrar data_encerramento. Rótulos dos filtros devem indicar qual data está sendo usada.
Status, responsável e perfil são atuais, mesmo em eventos de datas anteriores. Não há histórico de criação de usuários nem de mudanças de status. A série omite dias sem eventos; o dashboard pode completar o calendário com zeros.
As tabelas agregadas reduzem linhas e joins para gráficos e cards. A base detalhada preserva o grão para relatórios e estatísticas. Não se particiona por status, perfil ou responsável, evitando fragmentação em tabelas pequenas. Nenhuma configuração específica de cluster ou comando OPTIMIZE é exigido.
Todas as Gold incluem atualizado_em e versao_publicacao. Gravações Delta são atômicas por tabela, não entre tabelas: configurar o dashboard/Job após sucesso completo da Gold e conferir a mesma versao_publicacao ao combinar saídas. Executar Bronze → Silver → Gold sequencialmente, sem cargas concorrentes.
Nomes de responsáveis são dados pessoais: disponibilizar as Gold somente aos consumidores autorizados do Superadmin. CPF, email, senha e textos livres dos chamados não são publicados.
